# 01 — Bronze to Silver: Ingenieria de Datos
**Objetivo:** Transformar datos crudos (Bronze) en datasets limpios, filtrados y validados (Silver).

**Outputs:**
- `data/silver/wpp2024_countries_estimates.parquet`
- `data/silver/wpp2024_countries_projections.parquet`
- `data/silver/data_quality_report.json`

---


## 1.1 Imports y configuracion

In [29]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from src.config import BRONZE, SILVER, ESTIMATION_CUTOFF, COUNTRY_LOCTYPEID
from src.loaders import load_medium, load_supplement
from src.transforms import filter_countries, split_estimates_projections, create_derived_features, merge_supplement
from src.validators import audit_missing_values, validate_births_consistency, generate_quality_report

## 1.2 Cargar datos Bronze

In [30]:
df_medium = load_medium()
df_supplement = load_supplement()

print(f"Medium crudo: {df_medium.shape[0]:,} filas x {df_medium.shape[1]} columnas")
print(f"Suplemento:   {df_supplement.shape[0]:,} filas x {df_supplement.shape[1]} columnas")

Medium crudo: 84,360 filas x 67 columnas
Suplemento:   35,636 filas x 9 columnas


In [31]:
df_medium.columns


Index(['SortOrder', 'LocID', 'Notes', 'ISO3_code', 'ISO2_code', 'SDMX_code',
       'LocTypeID', 'LocTypeName', 'ParentID', 'Location', 'VarID', 'Variant',
       'Time', 'TPopulation1Jan', 'TPopulation1July', 'TPopulationMale1July',
       'TPopulationFemale1July', 'PopDensity', 'PopSexRatio', 'MedianAgePop',
       'NatChange', 'NatChangeRT', 'PopChange', 'PopGrowthRate',
       'DoublingTime', 'Births', 'Births1519', 'CBR', 'TFR', 'NRR', 'MAC',
       'SRB', 'Deaths', 'DeathsMale', 'DeathsFemale', 'CDR', 'LEx', 'LExMale',
       'LExFemale', 'LE15', 'LE15Male', 'LE15Female', 'LE65', 'LE65Male',
       'LE65Female', 'LE80', 'LE80Male', 'LE80Female', 'InfantDeaths', 'IMR',
       'LBsurvivingAge1', 'Under5Deaths', 'Q5', 'Q0040', 'Q0040Male',
       'Q0040Female', 'Q0060', 'Q0060Male', 'Q0060Female', 'Q1550',
       'Q1550Male', 'Q1550Female', 'Q1560', 'Q1560Male', 'Q1560Female',
       'NetMigrations', 'CNMR'],
      dtype='str')

In [32]:
df_supplement.columns

Index(['LocID', 'ISO3_code', 'Location', 'Time', 'BirthsMale', 'BirthsFemale',
       'LE60', 'LE60Male', 'LE60Female'],
      dtype='str')

## 1.3 Filtrar solo paises (LocTypeID == 4)
> Sin este filtro se produce doble conteo de poblacion.
> Ver [decision_log.md](../decisions/decision_log.md) — Seccion Notebook 01: *Filtrar por LocTypeID == 4 para analisis de paises*.

In [ ]:
df_countries = filter_countries(df_medium)
print(f"Medium filtrado a Localidades (LocTypeID == {COUNTRY_LOCTYPEID}): {df_countries.shape[0]:,} filas x {df_countries.shape[1]} columnas")
print(f"Localidades: {df_countries['LocID'].nunique()}")

Medium filtrado a paises (LocTypeID == 4): 36,024 filas x 67 columnas
Paises unicos: 237


In [42]:
# Codigos ISO3 de las 42 dependencias, territorios de ultramar y regiones especiales
non_sovereign_iso3 = [
    'ASM', 'AIA', 'ABW', 'BMU', 'BES', 'VGB', 'CYM', 'HKG', 'MAC', 'TWN',
    'COK', 'CUW', 'FLK', 'FRO', 'GUF', 'PYF', 'GIB', 'GRL', 'GLP', 'GUM',
    'GGY', 'IMN', 'JEY', 'XKX', 'MTQ', 'MYT', 'MSR', 'NCL', 'NIU', 'MNP',
    'PRI', 'REU', 'BLM', 'SHN', 'MAF', 'SPM', 'SXM', 'TKL', 'TCA', 'VIR',
    'WLF', 'ESH'
]

# Identificar y auditar las 42 entidades no soberanas
df_deps = df_countries[df_countries['ISO3_code'].isin(non_sovereign_iso3)][['Location', 'ISO3_code']].drop_duplicates().sort_values('Location')

n_total = df_countries['LocID'].nunique()
n_sovereign = n_total - len(df_deps)
print(f"Entidades totales (LocTypeID == 4): {n_total} ({n_sovereign} Paises Soberanos + {len(df_deps)} Localidades/Territorios dependientes)\n")
display(df_deps.reset_index(drop=True))


Entidades totales (LocTypeID == 4): 237 (195 Paises Soberanos + 42 Localidades/Territorios dependientes)



,Location,ISO3_code
0,American Samoa,ASM
1,Anguilla,AIA
2,Aruba,ABW
3,Bermuda,BMU
4,"Bonaire, Sint Eustatius and Saba",BES
5,British Virgin Islands,VGB
6,Cayman Islands,CYM
7,"China, Hong Kong SAR",HKG
8,"China, Macao SAR",MAC
9,"China, Taiwan Province of China",TWN


> **Rigor conceptual sobre las entidades analizadas:**
> * El dataset de la ONU (`LocTypeID == 4`) se titula oficialmente **"Country/Area"** e incluye 237 unidades: **195 Estados soberanos** (193 miembros ONU + 2 observadores) y **42 dependencias y territorios autónomos** (ej. Puerto Rico, Groenlandia, Hong Kong, Guayana Francesa).
> * **Decisión terminológica:** Para evitar inexactitudes geopolíticas en las interpretaciones y reportes, se adoptará la denominación oficial **"Países y Áreas"** (o **"Localidades"**), sin asumir que las 237 entidades son países soberanos independientes.
> * Ver [decision_log.md](../decisions/decision_log.md) — Sección Notebook 01: *Distincion politica y conceptual: Paises soberanos vs Localidades/Territorios dependientes*.


## 1.4 Separar estimaciones vs proyecciones
> Transparencia en supuestos: Las estimaciones y proyecciones tienen niveles de incertidumbre distintos.
> Ver [decision_log.md](../decisions/decision_log.md) — Seccion Notebook 01: *Separar estimaciones (<=2023) de proyecciones (>=2024)*.

In [34]:
df_est, df_proj = split_estimates_projections(df_countries)
print(f"Estimaciones (<= {ESTIMATION_CUTOFF}): {df_est.shape[0]:,} filas (rango {df_est['Time'].min()}-{df_est['Time'].max()})")
print(f"Proyecciones (> {ESTIMATION_CUTOFF}):  {df_proj.shape[0]:,} filas (rango {df_proj['Time'].min()}-{df_proj['Time'].max()})")

Estimaciones (<= 2023): 17,538 filas (rango 1950-2023)
Proyecciones (> 2023):  18,486 filas (rango 2024-2101)


## 1.5 Merge con Supplement (nacimientos por sexo + LE60)

In [35]:
df_est = merge_supplement(df_est, df_supplement)
df_proj = merge_supplement(df_proj, df_supplement)

print(f"Estimaciones post-merge: {df_est.shape[0]:,} filas x {df_est.shape[1]} columnas")
print(f"Proyecciones post-merge: {df_proj.shape[0]:,} filas x {df_proj.shape[1]} columnas")

Estimaciones post-merge: 17,538 filas x 72 columnas
Proyecciones post-merge: 18,486 filas x 72 columnas


> **Justificación del merge:**
> 1. **Sobrevida geriátrica (`LE60`):** La esperanza de vida al nacer (`LEx`) se ve arrastrada hacia abajo por la mortalidad infantil, lo que distorsiona la medición de la longevidad adulta. `LE60` aísla los años de vida esperados para quienes ya alcanzaron los 60 años, midiendo con precisión la salud en la vejez y el envejecimiento poblacional en la edad de retiro.
> 2. **Desglose de nacimientos por sexo (`BirthsMale`, `BirthsFemale`):** Permite identificar alteraciones estructurales o culturales en la razón de sexo al nacer (desviaciones del ratio biológico natural de ~105 varones por cada 100 mujeres) y sirve como control de calidad para verificar consistencia interna ($\text{Births} \approx \text{BirthsMale} + \text{BirthsFemale}$).


## 1.6 Auditoria de missings

In [36]:
# Auditoria de valores faltantes
missing_est = audit_missing_values(df_est, "Estimaciones")
display(missing_est)

,column,n_missing,pct_missing,dtype
0,Notes,12876,73.42,str
1,DoublingTime,3601,20.53,float64
2,ISO2_code,74,0.42,str
3,BirthsMale,74,0.42,float64
4,BirthsFemale,74,0.42,float64
5,LE60,74,0.42,float64
6,LE60Male,74,0.42,float64
7,LE60Female,74,0.42,float64


In [ ]:
# 1. Diagnostico: paises asociados a los nulos
locs_iso2_null = df_est[df_est["ISO2_code"].isnull()]["Location"].unique()
locs_suppl_null = df_est[df_est["LE60"].isnull()]["Location"].unique()
print("Entidad con ISO2_code nulo:", locs_iso2_null)
print("Entidad con suplemento nulo (LE60, Births por sexo):", locs_suppl_null)


Entidad con ISO2_code nulo: <ArrowStringArray>
['Namibia']
Length: 1, dtype: str
Entidad con suplemento nulo (LE60, Births por sexo): <ArrowStringArray>
['Holy See']
Length: 1, dtype: str


In [38]:
# 2. Tratamiento tecnico: Restaurar codigo ISO2 de Namibia ('NA')
df_est["ISO2_code"] = df_est["ISO2_code"].fillna("NA")
df_proj["ISO2_code"] = df_proj["ISO2_code"].fillna("NA")
print("Nulos restantes en ISO2_code:", int(df_est["ISO2_code"].isnull().sum()))

Nulos restantes en ISO2_code: 0


> **Interpretación de la auditoría y decisiones de calidad:**
> * **Completitud general:** De las 76 variables, 68 presentan 100% de datos completos (0% nulos), incluyendo todos los indicadores demográficos fundamentales (`TFR`, `LEx`, `IMR`, `CNMR`, `PopGrowthRate`).
> * **Nulos en `DoublingTime` (20.53%):** No corresponden a una falla técnica sino a una restricción matemática y demográfica. El tiempo de duplicación ($T_d = \ln(2)/r$) está indefinido cuando la tasa de crecimiento es nula o negativa ($r \le 0$). En poblaciones en declive o estancamiento (frecuentes en Europa y Asia oriental), la población nunca se duplicará, por lo que la ONU asigna `NaN`. Se excluye del índice compuesto para no perder el 20% de las observaciones y se adopta en su lugar **`PopGrowthRate`** (variable continua con 0% nulos).
> * **Corrección en `ISO2_code` (0.42% = 74 nulos):** Pertenecían exclusivamente a Namibia (`LocID: 516`) porque su código oficial es `"NA"`, interpretado automáticamente como valor faltante. Se corrigió imputando `"NA"` como texto literal.
> * **Nulos en variables del suplemento (0.42% = 74 nulos):** Pertenecen exclusivamente a la Santa Sede (`LocID: 336`, Vaticano), microestado de ~800 clérigos sin nacimientos locales. Se documenta para excluirlo del cálculo del índice multivariado y evitar distorsiones por poblaciones no representativas.
> * Ver [decision_log.md](../decisions/decision_log.md) — Sección Notebook 01: *Auditoria de completitud, tratamiento de nulos y exclusion de DoublingTime*.


## 1.7 Variables derivadas

In [39]:
df_est = create_derived_features(df_est)
df_proj = create_derived_features(df_proj)

cols_derived = ['GenderGapLE', 'IsEstimate', 'Epoch', 'NaturalGrowth']
display(df_est[['Location', 'Time'] + cols_derived].head())

,Location,Time,GenderGapLE,IsEstimate,Epoch,NaturalGrowth
0,Burundi,1950,2.6741,True,1950-1979,28.627
1,Burundi,1951,2.8195,True,1950-1979,27.581
2,Burundi,1952,2.8638,True,1950-1979,26.922
3,Burundi,1953,2.8269,True,1950-1979,26.593
4,Burundi,1954,2.8510,True,1950-1979,26.209


> **Justificación de las variables derivadas:**
> * **`GenderGapLE` ($\text{LExFemale} - \text{LExMale}$):** Mide la brecha de género en años de esperanza de vida. Captura inequidades de salud materna (cuando la brecha es baja o negativa) o sobremortalidad masculina adulta por factores de riesgo y conflictos (cuando supera los 8–10 años). Es una dimensión clave para el pilar de salud y equidad del índice.
> * **`NaturalGrowth` ($\text{CBR} - \text{CDR}$):** Tasa neta de crecimiento natural por 1.000 habitantes. Aísla la dinámica vegetativa interna de los flujos migratorios internacionales (`CNMR`).
> * **`IsEstimate` ($\text{Time} \le 2023$):** Distingue de forma explícita las observaciones históricas reconstruidas con censos/encuestas de las proyecciones y simulaciones probabilísticas futuras.
> * **`Epoch`:** Segmenta la serie en periodos homogéneos (`1950-1979`, `1980-1999`, `2000-2023`, `2024-2049`, `2050-2100`) para analizar trayectorias de transición demográfica sin atomizar el análisis año a año.
> * Ver [decision_log.md](../decisions/decision_log.md) — Sección Notebook 01: *Creacion de variables derivadas (GenderGapLE, IsEstimate, Epoch, NaturalGrowth)*.


## 1.8 Validacion de consistencia

In [40]:
inconsistencies = validate_births_consistency(df_est)
print(f"Inconsistencias encontradas en nacimientos (BirthsMale + BirthsFemale vs Births): {len(inconsistencies)}")
if len(inconsistencies) > 0:
    display(inconsistencies[['Location', 'Time', 'Births', 'BirthsMale', 'BirthsFemale', 'diff_births']].head())

Inconsistencias encontradas en nacimientos (BirthsMale + BirthsFemale vs Births): 89


,Location,Time,Births,BirthsMale,BirthsFemale,diff_births
4041,Saint Helena,1995,0.095,0.049,0.047,0.001
4042,Saint Helena,1996,0.089,0.045,0.043,0.001
4045,Saint Helena,1999,0.077,0.040,0.038,0.001
4047,Saint Helena,2001,0.071,0.036,0.034,0.001
4055,Saint Helena,2009,0.046,0.023,0.022,0.001


> **Interpretación del control de consistencia:**
> * El 99.5% de las observaciones presenta consistencia exacta entre nacimientos totales y desglosados por sexo.
> * Las 89 inconsistencias detectadas ocurren exclusivamente en microterritorios insulares (Santa Elena, Niue, Montserrat, Tokelau) con menos de 100 nacimientos anuales.
> * En todos los casos, la diferencia absoluta es de exactamente $0.001$ miles (1 solo nacimiento), atribuible al redondeo independiente a tres decimales en las publicaciones de la ONU.
> * **Conclusión:** No existen discrepancias estructurales ni errores de captura en el dataset.


## 1.9 Guardar en Silver (Parquet)

In [41]:
SILVER.mkdir(parents=True, exist_ok=True)
est_path = SILVER / 'wpp2024_countries_estimates.parquet'
proj_path = SILVER / 'wpp2024_countries_projections.parquet'
report_path = SILVER / 'data_quality_report.json'

df_est.to_parquet(est_path, index=False)
df_proj.to_parquet(proj_path, index=False)

quality_rep = generate_quality_report(df_est, output_path=report_path)
print(f"Estimaciones guardadas en: {est_path} ({round(est_path.stat().st_size / (1024*1024), 2)} MB)")
print(f"Proyecciones guardadas en: {proj_path} ({round(proj_path.stat().st_size / (1024*1024), 2)} MB)")
print('Reporte de calidad generado:')
for k, v in quality_rep.items():
    print(f" - {k:18s}: {v}")

Estimaciones guardadas en: C:\Users\USUARIO\Documents\Universidade_2026-2S\ASA\2026 ASA DataQuest\data\silver\wpp2024_countries_estimates.parquet (6.75 MB)
Proyecciones guardadas en: C:\Users\USUARIO\Documents\Universidade_2026-2S\ASA\2026 ASA DataQuest\data\silver\wpp2024_countries_projections.parquet (6.71 MB)
Reporte de calidad generado:
 - n_rows            : 17538
 - n_cols            : 76
 - memory_mb         : 10.52
 - n_duplicates      : 0
 - columns_with_na   : 7
 - total_na_cells    : 16847
 - na_pct            : 1.26


---
**Silver listo. Continua con `02_eda_univariate.ipynb`.**
